In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Load the dataset
df = pd.read_csv("salary.csv")

# Split into features and target
X = df[["YearsExperience"]]
y = df["Salary"]

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
import joblib

loaded_model = joblib.load("salary_model.joblib")
loaded_model.predict(X_test)

array([114509.02      ,  63907.88      , 103274.55      ,  70153.12      ,
        55881.11666667,  60478.69269048])

See "saving2.ipynb" for code

The predictions are exactly the same. They match because its the same model just loaded from a saved state.

Task 5: Log Your Baseline Run with MLflow

In [10]:
import mlflow

with mlflow.start_run():
    mlflow.log_param("n_estimatores", 100)
    predictions = loaded_model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    mlflow.log_metric("mae", mae)

    print("logged run with MAE", round(mae, 2))


logged run with MAE 6872.01


Task 6: Tweak a Hyperparameter and Compare

In [11]:
X = df[["YearsExperience"]]
y = df["Salary"]

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train a Random Forest model with default settings
model = RandomForestRegressor(n_estimators=10, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print("MAE:", mae)

with mlflow.start_run():
    mlflow.log_param("n_estimatores", 10)
    mlflow.log_metric("mae", mae)

    print(round(mae, 2))

MAE: 7262.783333333334
7262.78


Task 7: Compare Runs in the MLflow UI

the 100 n_estimator performed better giving an MAE of: 6872 
vs
10 at: 7262

the n_estimators controls the number of trees in the decision tree. More trees with decision nodes split it more giving better accuracy.

Task 8: Run a Hyperparameter Sweep

In [16]:
X = df[["YearsExperience"]]
y = df["Salary"]

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train a Random Forest model with default settings
model = RandomForestRegressor(n_estimators=175, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print("MAE:", mae)

with mlflow.start_run():
    mlflow.log_param("n_estimatores", 175)
    mlflow.log_metric("mae", mae)

    print(round(mae, 2))

MAE: 6752.299145124717
6752.3


1. more trees did not mean better MAE
2. for our runs, 150 was best. 175 dropped, indicating 150 was the point where increasing estimators stopped helping.
3. Time wasn't an issue here, improvement on this dataset would likely be worth it.

best run was n_estimators=150


Task 9: Research Model Compatibility Risks

Code crashes and broken pipelines may occur if joblib files are loaded much later on your models. 
Joblib serializes Python objects directly, relying on the exact internal structure of the libraries installed at the time of saving.

To prevent this: setup the joblib file as you start tweaking your model and record the library versions and environments in the project folders.

Task 10: Build a Reusable Train-and-Log Function

In [22]:
def train_log(df, feature_col, target_col, best_mae=float("inf")):
    X = df[[feature_col]]
    y = df[target_col]
        
    X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=175, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)

    with mlflow.start_run():
        mlflow.log_param("n_estimatores", 175)
        mlflow.log_metric("mae", mae)

    if mae < best_mae:
        joblib.dump(model, "best_model.joblib")
        best_mae = mae
    else:
        print("You have best mae")
        
    return mae, best_mae


In [24]:
best_mae = float("inf")

print(train_log(df, "YearsExperience", "Salary", best_mae))

(6752.299145124717, 6752.299145124717)
